# 01 Data preparation

This is the only notebook that reads raw inputs: the NSO Excel workbooks, the ERA5 monthly 2 m temperature GRIB, the ERA5-Drought SPEI-3 archive and the aimag GeoJSON. It writes every derived table that notebooks 02-05 consume into `data/processed/`.

The derived CSVs are committed to the repository.

In [ ]:
import sys, os
from pathlib import Path

REPO = "mongolia-dzud-drought"
IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if IN_COLAB and not Path(REPO).exists() and Path.cwd().name != REPO:
    !git clone -q https://github.com/nominkhurelchuluun/{REPO}.git /content/{REPO}
    os.chdir(f"/content/{REPO}")
elif Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.insert(0, str(Path.cwd()))

!pip install -q cfgrib eccodes mapclassify netCDF4

import numpy as np
import pandas as pd
import xarray as xr

from src import data_loading as dl
dl.describe_paths()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 818.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.6/91.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 45.0 MB/s eta 0:00:00
DATA_DIR      /content/mongolia-dzud-drought/data
RAW_DIR       /content/mongolia-dzud-drought/data/raw
PROCESSED_DIR /content/mongolia-dzud-drought/data/processed
FIG_DIR       /content/mongolia-dzud-drought/figures


In [ ]:
dl.use_colab_drive()
dl.describe_paths()

Mounted at /content/drive
DATA_DIR      /content/drive/My Drive/mongolia-dzud-drought/data
RAW_DIR       /content/drive/My Drive/mongolia-dzud-drought/data/raw
PROCESSED_DIR /content/drive/My Drive/mongolia-dzud-drought/data/processed
FIG_DIR       /content/drive/My Drive/mongolia-dzud-drought/figures


In [ ]:
!pip install -q netCDF4

In [ ]:
!python scripts/preflight.py


0. Paths
DATA_DIR      /content/mongolia-dzud-drought/data
RAW_DIR       /content/mongolia-dzud-drought/data/raw
PROCESSED_DIR /content/mongolia-dzud-drought/data/processed
FIG_DIR       /content/mongolia-dzud-drought/figures
[FAIL] mortality: mongolia_mortality_rates.xlsx not found in /content/mongolia-dzud-drought/data/raw
[FAIL] population: types_livestock_aimag.xlsm not found in /content/mongolia-dzud-drought/data/raw
[FAIL] aimags: mongolia_adm1_geoboundaries.geojson not found in /content/mongolia-dzud-drought/data/raw
[FAIL] t2m_grib: era5_monthly_t2m_mongolia_1950_2024.grib not found in /content/mongolia-dzud-drought/data/raw
[FAIL] spei_zip: era5_drought_spei3_mongolia_1950_2024.zip not found in /content/mongolia-dzud-drought/data/raw

Place the missing files in data/raw/ using the names above, or edit RAW_FILES in src/data_loading.py to match your filenames.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import xarray as xr

from src import data_loading as dl

dl.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
dl.SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
dl.describe_paths()

DATA_DIR      /content/drive/My Drive/mongolia-dzud-drought/data
RAW_DIR       /content/drive/My Drive/mongolia-dzud-drought/data/raw
PROCESSED_DIR /content/drive/My Drive/mongolia-dzud-drought/data/processed
FIG_DIR       /content/drive/My Drive/mongolia-dzud-drought/figures


## 1. Livestock mortality workbook

`mongolia_mortality_rates.xlsx` holds one row per aimag, year and species with the reported death count, the prior-year population and the derived mortality rate.

`All_SFU` is computed here as Equation (2): the ratio of SFU-weighted deaths to SFU-weighted prior-year population. It is **not** an SFU-weighted average of the five species rates.

In [ ]:
import pandas as pd
import numpy as np
mortality_raw = pd.read_excel(dl.raw_path("mortality"))
mortality_raw.columns = [str(c).strip() for c in mortality_raw.columns]

DEATH_COL = next(c for c in mortality_raw.columns if "death" in c.lower())
PRIOR_COL = next(c for c in mortality_raw.columns
                 if "prior" in c.lower() and "year" in c.lower())
RATE_COL = next(c for c in mortality_raw.columns
                if "mortality" in c.lower() and "rate" in c.lower())

mortality_raw["Year"] = pd.to_numeric(mortality_raw["Year"], errors="coerce")
mortality_raw = mortality_raw.dropna(subset=["Year"])
mortality_raw["Year"] = mortality_raw["Year"].astype(int)
mortality_raw["Aimag"] = mortality_raw["Aimag"].astype(str).str.strip()
mortality_raw["Species"] = mortality_raw["Species"].astype(str).str.strip()

for column in (DEATH_COL, PRIOR_COL, RATE_COL):
    mortality_raw[column] = pd.to_numeric(mortality_raw[column], errors="coerce")

mortality_raw["SFU"] = mortality_raw["Species"].map(dl.SFU_BY_NAME)
if mortality_raw["SFU"].isna().any():
    raise KeyError("Unmapped species labels: "
                   + ", ".join(sorted(mortality_raw.loc[mortality_raw["SFU"].isna(),
                                                        "Species"].unique())))

mortality_raw["Loss_SFU"] = mortality_raw[DEATH_COL] * mortality_raw["SFU"]
mortality_raw["Denom_SFU"] = mortality_raw[PRIOR_COL] * mortality_raw["SFU"]

print(f"{len(mortality_raw)} species-aimag-year rows, "
      f"{mortality_raw['Year'].min()}-{mortality_raw['Year'].max()}, "
      f"{mortality_raw['Aimag'].nunique()} aimags")

3465 species-aimag-year rows, 1992-2024, 21 aimags


In [ ]:
species_rates = (mortality_raw
                 .pivot_table(index=["Aimag", "Year"], columns="Species",
                              values=RATE_COL, aggfunc="mean")
                 .rename(columns=dl.SPECIES_MAP)
                 .reset_index())
species_rates.columns.name = None
species_rates = species_rates.reindex(
    columns=["Aimag", "Year"] + dl.SPECIES_COLS)

sfu_rates = (mortality_raw.groupby(["Aimag", "Year"], as_index=False)
             .agg(Loss_SFU=("Loss_SFU", "sum"), Denom_SFU=("Denom_SFU", "sum")))
sfu_rates["All_SFU"] = sfu_rates["Loss_SFU"] / sfu_rates["Denom_SFU"]

mortality_rates = species_rates.merge(
    sfu_rates[["Aimag", "Year", "All_SFU"]], on=["Aimag", "Year"], how="left")
mortality_rates["All"] = mortality_rates[dl.SPECIES_COLS].mean(axis=1)
mortality_rates = mortality_rates[
    mortality_rates["Year"].between(dl.MORT_START, dl.MORT_END)].reset_index(drop=True)

for column in dl.SPECIES_COLS + ["All", "All_SFU"]:
    mortality_rates[column] = mortality_rates[column].replace([np.inf, -np.inf], np.nan)

naive_sfu_mean = (mortality_rates[dl.SPECIES_COLS]
                  .mul(pd.Series(dl.SFU_WEIGHTS))
                  .sum(axis=1, min_count=1)
                  / mortality_rates[dl.SPECIES_COLS].notna()
                  .mul(pd.Series(dl.SFU_WEIGHTS)).sum(axis=1))

print("Equation (2) SFU rate vs SFU-weighted average of species rates")
print(f"  mean absolute difference {np.nanmean(np.abs(mortality_rates['All_SFU'] - naive_sfu_mean)) * 100:.3f} pp")
print(f"  max absolute difference  {np.nanmax(np.abs(mortality_rates['All_SFU'] - naive_sfu_mean)) * 100:.3f} pp")
print(f"  national mean, Equation (2)          {mortality_rates['All_SFU'].mean() * 100:.2f}%")
print(f"  national mean, weighted rate average {naive_sfu_mean.mean() * 100:.2f}%")

mortality_rates.to_csv(dl.processed_path("mortality_rates_aimag.csv"), index=False)
MORT_AIMAGS = sorted(mortality_rates["Aimag"].unique())
print(f"\n{len(MORT_AIMAGS)} aimags in the mortality record")

Equation (2) SFU rate vs SFU-weighted average of species rates
  mean absolute difference 0.843 pp
  max absolute difference  18.784 pp
  national mean, Equation (2)          4.48%
  national mean, weighted rate average 4.13%

21 aimags in the mortality record


## 2. Absolute losses

Annual national deaths by species, in millions of head.

In [ ]:
losses_annual = (mortality_raw.pivot_table(index="Year", columns="Species",
                                           values=DEATH_COL, aggfunc="sum")
                 .sort_index() / 1000.0)
losses_annual = losses_annual.loc[dl.MORT_START:dl.MORT_END]
losses_annual["Total"] = losses_annual.sum(axis=1)
losses_annual.reset_index().to_csv(
    dl.processed_path("livestock_losses_national_annual.csv"), index=False)

print(losses_annual.round(2).to_string())

Species  Camel  Cattle  Goat  Horse  Sheep  Total
Year                                             
1992      0.02    0.12  0.13   0.08   0.51   0.86
1993      0.02    0.27  0.28   0.14   0.92   1.64
1994      0.01    0.08  0.14   0.04   0.42   0.68
1995      0.01    0.08  0.13   0.05   0.40   0.66
1996      0.01    0.08  0.15   0.05   0.28   0.58
1997      0.01    0.11  0.18   0.04   0.28   0.61
1998      0.01    0.11  0.27   0.05   0.34   0.78
1999      0.01    0.12  0.24   0.07   0.33   0.77
2000      0.02    0.63  1.03   0.39   1.41   3.48
2001      0.02    1.01  1.28   0.45   1.98   4.73
2002      0.03    0.20  1.31   0.21   1.17   2.91
2003      0.01    0.21  0.43   0.14   0.53   1.32
2004      0.00    0.05  0.10   0.03   0.12   0.29
2005      0.00    0.06  0.32   0.04   0.25   0.67
2006      0.00    0.04  0.22   0.03   0.18   0.47
2007      0.00    0.03  0.12   0.02   0.12   0.29
2008      0.00    0.13  0.73   0.12   0.65   1.64
2009      0.00    0.11  0.88   0.06   0.68   1.73


## 3. Livestock population workbook

`types_livestock_aimag.xlsm` is a wide sheet with a livestock-type column, an aimag column and one column per year, in thousands of head.

In [ ]:
population_raw = pd.read_excel(dl.raw_path("population"))
population_raw["Livestock_Type"] = population_raw["Unnamed: 0"].ffill()
population_raw["Aimag"] = population_raw["Unnamed: 1"].ffill()
population_raw = population_raw.drop(columns=["Unnamed: 0", "Unnamed: 1"])

year_columns = [c for c in population_raw.columns if str(c).strip().isdigit()]

population_long = population_raw.melt(
    id_vars=["Livestock_Type", "Aimag"], value_vars=year_columns,
    var_name="Year", value_name="Value")
population_long["Year"] = pd.to_numeric(population_long["Year"], errors="coerce")
population_long["Value"] = pd.to_numeric(population_long["Value"], errors="coerce")
population_long = population_long.dropna(subset=["Year", "Value"])
population_long["Year"] = population_long["Year"].astype(int)
population_long["_type"] = (population_long["Livestock_Type"].astype(str)
                            .str.strip().str.lower())
population_long["_aimag"] = (population_long["Aimag"].astype(str)
                             .str.strip().str.lower())

national_total = (population_long[population_long["_type"].eq("total")
                                  & population_long["_aimag"].eq("total")]
                  .assign(Population_millions=lambda d: d["Value"] / 1000)
                  [["Year", "Population_millions"]]
                  .drop_duplicates(subset="Year", keep="first")
                  .sort_values("Year").reset_index(drop=True))
national_total["Change_millions"] = national_total["Population_millions"].diff()
national_total["Change_percent"] = (national_total["Population_millions"]
                                    .pct_change() * 100)
national_total.to_csv(
    dl.processed_path("livestock_population_national_total.csv"), index=False)

national_species = {}
for key in dl.SFU_BY_SPECIES:
    subset = population_long[population_long["_aimag"].eq("total")
                             & population_long["_type"].str.contains(key, na=False)
                             & ~population_long["_type"].str.contains("total", na=False)]
    if subset.empty:
        raise KeyError(f"No national population series matched '{key}'")
    national_species[key.capitalize()] = (subset.groupby("Year")["Value"].sum()
                                          .sort_index() / 1000.0)

national_species = pd.DataFrame(national_species)
national_species["Headcount"] = national_species.sum(axis=1)
national_species["SFU_units"] = sum(
    national_species[name] * weight for name, weight in dl.SFU_BY_NAME.items())
national_species.reset_index().to_csv(
    dl.processed_path("livestock_population_national_species.csv"), index=False)

print(national_species.round(2).tail().to_string())

      Horse  Cattle  Camel  Sheep   Goat  Headcount  SFU_units
Year                                                          
2020   4.09    4.73   0.47  30.05  27.72      67.07     114.41
2021   4.32    5.02   0.45  31.09  26.46      67.34     117.57
2022   4.82    5.51   0.47  32.75  27.57      71.12     126.74
2023   4.83    5.35   0.47  29.41  24.62      64.68     119.85
2024   4.68    5.07   0.48  24.49  22.92      57.65     110.75


In [ ]:
aimag_species = population_long[
    ~population_long["_aimag"].eq("total")
    & ~population_long["_type"].str.contains("total", na=False)].copy()
aimag_species["Species"] = aimag_species["_type"].str.capitalize()
aimag_species["Aimag"] = (aimag_species["Aimag"].astype(str).str.strip()
                          .replace(dl.RENAME_GEO))
aimag_species = (aimag_species.groupby(["Aimag", "Year", "Species"], as_index=False)
                 ["Value"].sum()
                 .rename(columns={"Value": "Thousand_head"}))
aimag_species.to_csv(
    dl.processed_path("livestock_population_aimag_species.csv"), index=False)

print(f"{len(aimag_species)} aimag-year-species population rows")

7532 aimag-year-species population rows


## 4. Aimag boundaries

Boundaries are the geoBoundaries ADM1 layer for Mongolia (Runfola et al., 2020), derived from OpenStreetMap under ODbL 1.0. The retrieval writes the API response to a .SOURCE.json sidecar, and the pinned commit hash in the download URL makes it reproducible. Ulaanbaatar is dropped as a municipality, leaving the 21 units in the mortality record; names are harmonised to NSO spellings via RENAME_GEO.

In [ ]:
import geopandas as gpd
import requests, json
from datetime import date

gb_path = dl.raw_path("aimags")
sidecar = gb_path.with_suffix("").with_suffix(".SOURCE.json")

if not gb_path.exists():
    meta = requests.get(
        "https://www.geoboundaries.org/api/current/gbOpen/MNG/ADM1/").json()
    gpd.read_file(meta["gjDownloadURL"]).to_file(gb_path, driver="GeoJSON")
    sidecar.write_text(json.dumps(
        {"retrieved": date.today().isoformat(),
         "api": "https://www.geoboundaries.org/api/current/gbOpen/MNG/ADM1/",
         "metadata": meta}, indent=2))

info = json.loads(sidecar.read_text())["metadata"]
print(info["boundaryName"], info["boundaryYearRepresented"],
      info["licenseDetail"])
print(info["gjDownloadURL"])

Mongolia 2017 Open Data Commons Open Database License 1.0
https://github.com/wmgeolab/geoBoundaries/raw/9469f09/releaseData/gbOpen/MNG/ADM1/geoBoundaries-MNG-ADM1.geojson


In [ ]:
aimags = dl.load_aimag_boundaries(dissolve=True)
aimags.to_file(dl.processed_path("aimag_boundaries.geojson"), driver="GeoJSON")

assert aimags.crs.to_string() == "EPSG:4326", aimags.crs
assert len(aimags) == 21, len(aimags)
assert aimags.geometry.is_valid.all(), "invalid geometry after dissolve"
assert set(aimags["Aimag"]) == set(MORT_AIMAGS), (
    sorted(set(MORT_AIMAGS) - set(aimags["Aimag"])),
    sorted(set(aimags["Aimag"]) - set(MORT_AIMAGS)))

print(f"{len(aimags)} aimag polygons, all valid, names reconcile")

21 aimag polygons, all valid, names reconcile


## 5. SPEI-3

One archive supplies every SPEI-3 value in the manuscript. Aimag and national means are area-weighted by the overlap between each grid cell and each aimag polygon.

A point-sampled variant is also written. It reproduces the aggregation method used in the submitted panel model, which assigned whole grid cells to the aimag containing the cell centroid and then took an unweighted arithmetic mean. Both variants use the current boundary source, so the point series differs from the submitted values by the boundary change as well as the aggregation method. Notebooks 02-05 use the area-weighted series by default.

In [ ]:
spei_files = dl.extract_spei_archive()
print(f"{len(spei_files)} SPEI-3 NetCDF files")

with xr.open_dataset(spei_files[0], engine=dl.netcdf_engine()) as probe:
    lat_name = [c for c in probe.coords if "lat" in c.lower()][0]
    lon_name = [c for c in probe.coords if "lon" in c.lower()][0]
    grid_lats = np.asarray(probe[lat_name].values, dtype=float)
    grid_lons = np.asarray(probe[lon_name].values, dtype=float)

cell_weights = dl.build_cell_area_weights(grid_lats, grid_lons, aimags)
cell_weights.to_csv(dl.processed_path("spei3_cell_area_weights.csv"), index=False)

print(f"{len(grid_lats)} x {len(grid_lons)} grid, "
      f"{len(cell_weights)} cell-aimag overlaps, "
      f"{cell_weights['Aimag'].nunique()} aimags")

913 SPEI-3 NetCDF files
43 x 129 grid, 3707 cell-aimag overlaps, 21 aimags


In [ ]:
spei_cells = dl.read_spei_months(spei_files)
spei_cells = spei_cells[spei_cells["Year"].between(dl.SPEI_START, dl.SPEI_END)]

spei_jja_cells = spei_cells[spei_cells["Month"].isin(dl.JJA_MONTHS)]

spei_jja_aimag = (dl.area_weighted_mean(spei_jja_cells, cell_weights, ["Aimag", "Year"])
                  .rename(columns={"SPEI": "JJA_Mean_SPEI3"})
                  .sort_values(["Aimag", "Year"]).reset_index(drop=True))
spei_jja_aimag.to_csv(
    dl.processed_path("spei3_jja_aimag_1950_2024.csv"), index=False)

spei_jja_national = (dl.area_weighted_mean(spei_jja_cells, cell_weights, ["Year"])
                     .rename(columns={"SPEI": "JJA_Mean_SPEI3"})
                     .sort_values("Year").reset_index(drop=True))
spei_jja_national.to_csv(
    dl.processed_path("spei3_jja_national_1950_2024.csv"), index=False)

spei_seasonal_cells = spei_cells.copy()
spei_seasonal_cells["Season"] = spei_seasonal_cells["Month"].map(dl.SEASON_MAP)
spei_seasonal_cells["Season_Year"] = np.where(
    spei_seasonal_cells["Month"].eq(12),
    spei_seasonal_cells["Year"] + 1, spei_seasonal_cells["Year"])

spei_seasonal_aimag = (
    dl.area_weighted_mean(spei_seasonal_cells, cell_weights,
                          ["Aimag", "Season_Year", "Season"])
    .rename(columns={"Season_Year": "Year", "SPEI": "Seasonal_SPEI3"})
    .sort_values(["Aimag", "Year", "Season"]).reset_index(drop=True))
spei_seasonal_aimag.to_csv(
    dl.processed_path("spei3_seasonal_aimag_1950_2024.csv"), index=False)

print(spei_jja_national.tail().round(3).to_string(index=False))

 Year  JJA_Mean_SPEI3
 2020          -0.428
 2021           0.174
 2022          -1.260
 2023          -0.322
 2024          -0.454


In [ ]:
point_lookup = cell_weights.sort_values("weight", ascending=False).drop_duplicates(
    subset=["latitude", "longitude"])[["latitude", "longitude", "Aimag"]]

spei_jja_point = (spei_jja_cells.merge(point_lookup, on=["latitude", "longitude"],
                                       how="inner")
                  .groupby(["Aimag", "Year"], as_index=False)["SPEI"].mean()
                  .rename(columns={"SPEI": "JJA_Mean_SPEI3_point"}))
spei_jja_point.to_csv(
    dl.processed_path("spei3_jja_aimag_point_1950_2024.csv"), index=False)

comparison = spei_jja_aimag.merge(spei_jja_point, on=["Aimag", "Year"])
difference = comparison["JJA_Mean_SPEI3"] - comparison["JJA_Mean_SPEI3_point"]
print("Area-weighted minus point-sampled JJA SPEI-3")
print(f"  mean {difference.mean():+.4f}  sd {difference.std():.4f}  "
      f"max |diff| {difference.abs().max():.4f}")

Area-weighted minus point-sampled JJA SPEI-3
  mean +0.0033  sd 0.0277  max |diff| 0.1487


In [ ]:
from scipy import stats

trend_rows = []
for period_start, period_end in [(dl.SPEI_START, dl.SPEI_END),
                                 (dl.MORT_START, dl.MORT_END)]:
    window = spei_jja_aimag[spei_jja_aimag["Year"].between(period_start, period_end)]
    for aimag, group in window.groupby("Aimag"):
        group = group.dropna(subset=["Year", "JJA_Mean_SPEI3"])
        if len(group) < 2:
            continue
        fit = stats.linregress(group["Year"], group["JJA_Mean_SPEI3"])
        trend_rows.append({
            "Aimag": aimag,
            "Period": f"{period_start}-{period_end}",
            "N_years": len(group),
            "SPEI_slope_per_decade": fit.slope * 10,
            "Drying_rate_per_decade": -fit.slope * 10,
            "r": fit.rvalue,
            "p_value": fit.pvalue,
            "standard_error_per_year": fit.stderr,
        })

spei_trends = (pd.DataFrame(trend_rows)
               .sort_values(["Period", "Drying_rate_per_decade"],
                            ascending=[True, False])
               .reset_index(drop=True))
spei_trends.to_csv(dl.processed_path("spei3_jja_aimag_trends.csv"), index=False)

for period, group in spei_trends.groupby("Period"):
    window = spei_jja_national[
        spei_jja_national["Year"].between(*[int(v) for v in period.split("-")])]
    fit = stats.linregress(window["Year"], window["JJA_Mean_SPEI3"])
    print(f"{period} national area-weighted trend "
          f"{fit.slope * 10:+.4f} SPEI units per decade, "
          f"r = {fit.rvalue:.3f}, p = {fit.pvalue:.3g}, n = {len(window)}")

1950-2024 national area-weighted trend -0.1507 SPEI units per decade, r = -0.554, p = 2.52e-07, n = 75
1992-2024 national area-weighted trend -0.3598 SPEI units per decade, r = -0.570, p = 0.000535, n = 33


## 6. ERA5 temperature

A single monthly 2 m temperature GRIB feeds every temperature number in the
manuscript. Grid points are assigned to the aimag whose polygon contains them, then averaged within aimag and season. December is assigned to the following DJF year, so DJF of year t spans December t-1 through February t.

Two winter Z-scores are written. `Temp_Winter_Z_study` standardises within aimag over 1992-2024 and reproduces the submitted figures. `Temp_Winter_Z_climate` standardises over the full 1950-2024 ERA5 record and is the climatological reference.

In [ ]:
from shapely.geometry import Point

temperature_ds = xr.open_dataset(dl.raw_path("t2m_grib"), engine="cfgrib")

temperature_var = next(v for v in temperature_ds.data_vars
                       if any(k in v.lower() for k in ("t2m", "temp", "t2")))
time_name = ("time" if "time" in temperature_ds[temperature_var].dims
             else "valid_time")

grid_lon, grid_lat = np.meshgrid(temperature_ds.longitude.values,
                                 temperature_ds.latitude.values)

grid_points = gpd.GeoDataFrame(
    {"latitude": grid_lat.ravel(), "longitude": grid_lon.ravel()},
    geometry=[Point(x, y) for x, y in zip(grid_lon.ravel(), grid_lat.ravel())],
    crs="EPSG:4326")

point_aimag = (gpd.sjoin(grid_points, aimags, how="left", predicate="within")
               [["latitude", "longitude", "Aimag"]]
               .dropna(subset=["Aimag"])
               .drop_duplicates(["latitude", "longitude"])
               .reset_index(drop=True))

print(f"{temperature_var} on a {grid_lat.shape} grid, "
      f"{len(point_aimag)} points inside an aimag")

t2m on a (43, 130) grid, 2936 points inside an aimag


In [ ]:
temperature = (temperature_ds[temperature_var].to_dataframe().reset_index()
               .rename(columns={temperature_var: "Temp_K"}))
temperature["date"] = pd.to_datetime(temperature[time_name], errors="coerce")
temperature = temperature.dropna(subset=["date", "Temp_K"])
temperature["Month"] = temperature["date"].dt.month
temperature["Calendar_Year"] = temperature["date"].dt.year
temperature["Temp_C"] = temperature["Temp_K"] - 273.15
temperature["Season"] = temperature["Month"].map(dl.SEASON_MAP)
temperature["Year"] = np.where(temperature["Month"].eq(12),
                               temperature["Calendar_Year"] + 1,
                               temperature["Calendar_Year"])

temperature = temperature.merge(point_aimag, on=["latitude", "longitude"], how="inner")

temperature_seasonal = (temperature.groupby(["Aimag", "Year", "Season"], as_index=False)
                        ["Temp_C"].mean()
                        .rename(columns={"Temp_C": "Seasonal_Temp_C"}))
temperature_seasonal.to_csv(
    dl.processed_path("era5_seasonal_temperature_aimag.csv"), index=False)

temperature_ds.close()

winter = (temperature_seasonal[temperature_seasonal["Season"].eq("DJF")]
          [["Aimag", "Year", "Seasonal_Temp_C"]]
          .rename(columns={"Seasonal_Temp_C": "Temp_Winter"})
          .sort_values(["Aimag", "Year"]).reset_index(drop=True))

study_window = winter["Year"].between(dl.MORT_START, dl.MORT_END)
study_stats = (winter[study_window].groupby("Aimag")["Temp_Winter"]
               .agg(study_mean="mean", study_sd=lambda s: s.std(ddof=1)))
climate_stats = (winter.groupby("Aimag")["Temp_Winter"]
                 .agg(climate_mean="mean", climate_sd=lambda s: s.std(ddof=1)))

winter = winter.merge(study_stats, on="Aimag").merge(climate_stats, on="Aimag")
winter["Temp_Winter_Z_study"] = ((winter["Temp_Winter"] - winter["study_mean"])
                                 / winter["study_sd"])
winter["Temp_Winter_Z_climate"] = ((winter["Temp_Winter"] - winter["climate_mean"])
                                   / winter["climate_sd"])
winter = winter.drop(columns=["study_mean", "study_sd",
                              "climate_mean", "climate_sd"])
winter.to_csv(dl.processed_path("era5_djf_temperature_aimag.csv"), index=False)

overlap = winter[study_window].dropna(
    subset=["Temp_Winter_Z_study", "Temp_Winter_Z_climate"])
print("Severe-cold classification, 1992-2024")
print(f"  Z < -1 under the 1992-2024 reference: "
      f"{(overlap['Temp_Winter_Z_study'] < dl.SEVERE_COLD_THRESHOLD).sum()} aimag-years")
print(f"  Z < -1 under the 1950-2024 reference: "
      f"{(overlap['Temp_Winter_Z_climate'] < dl.SEVERE_COLD_THRESHOLD).sum()} aimag-years")

Severe-cold classification, 1992-2024
  Z < -1 under the 1992-2024 reference: 122 aimag-years
  Z < -1 under the 1950-2024 reference: 109 aimag-years


## 7. Analysis panel

JJA SPEI-3 in year t-1 is paired with mortality reported in year t and with the DJF temperature Z-score spanning December t-1 to February t. The result is the n = 693 panel used by notebooks 03-05.

In [ ]:
spei_previous = (spei_jja_aimag.rename(columns={"JJA_Mean_SPEI3": "SPEI_PrevSummer"})
                 .assign(Year=lambda d: d["Year"] + 1))
spei_previous_point = (spei_jja_point
                       .rename(columns={"JJA_Mean_SPEI3_point": "SPEI_PrevSummer_point"})
                       .assign(Year=lambda d: d["Year"] + 1))

panel = (mortality_rates
         .merge(spei_previous, on=["Aimag", "Year"], how="inner")
         .merge(spei_previous_point, on=["Aimag", "Year"], how="left")
         .merge(winter, on=["Aimag", "Year"], how="left")
         .sort_values(["Aimag", "Year"])
         .reset_index(drop=True))

panel = panel[panel["Year"].between(dl.MORT_START, dl.MORT_END)].reset_index(drop=True)

expected_rows = 21 * (dl.MORT_END - dl.MORT_START + 1)
print(f"panel: {len(panel)} rows, {panel['Aimag'].nunique()} aimags, "
      f"{panel['Year'].min()}-{panel['Year'].max()}")
if len(panel) != expected_rows:
    print(f"WARNING expected {expected_rows} rows")

print("\nmissing values per column")
print(panel.isna().sum().loc[lambda s: s > 0].to_string()
      or "  none")

panel.to_csv(dl.processed_path("mortality_panel.csv"), index=False)

panel: 693 rows, 21 aimags, 1992-2024

missing values per column
Mort_cam    17


## 8. Manifest

Everything notebooks 02-05 read is listed below. If any file is absent, rerun the section above that writes it.

In [ ]:
for path in sorted(dl.PROCESSED_DIR.iterdir()):
    size_kb = path.stat().st_size / 1024
    print(f"{path.name:52s} {size_kb:9.1f} kB")

README.md                                                  2.3 kB
aimag_boundaries.geojson                                 594.3 kB
era5_djf_temperature_aimag.csv                            85.3 kB
era5_seasonal_temperature_aimag.csv                      170.7 kB
livestock_losses_national_annual.csv                       2.2 kB
livestock_population_aimag_species.csv                   198.4 kB
livestock_population_national_species.csv                  4.2 kB
livestock_population_national_total.csv                    2.9 kB
mortality_panel.csv                                      166.2 kB
mortality_rates_aimag.csv                                 99.5 kB
spei3_cell_area_weights.csv                              140.3 kB
spei3_jja_aimag_1950_2024.csv                             51.5 kB
spei3_jja_aimag_point_1950_2024.csv                       51.6 kB
spei3_jja_aimag_trends.csv                                 5.2 kB
spei3_jja_national_1950_2024.csv                           1.8 kB
spei3_seas